# Create CDMRP (Congressionally Directed Medical Research Programs) Awards

Creates DoD CDMRP grant awards from the Dimensions-for-DTIC public instance (dtic.dimensions.ai). ~23.7K grants.

**Prerequisites:**
- Run `scripts/local/cdmrp_to_s3.py` to download and upload the data first.

**Data source:** https://dtic.dimensions.ai (CDMRP funder grid.496791.4; cdmrp.health.mil decommissioned 2025-09)
**S3 location:** `s3a://openalex-ingest/awards/cdmrp/cdmrp_grants.parquet`

**CDMRP funder:**
- funder_id: 4320338273
- display_name: Congressionally Directed Medical Research Programs

**Data notes:**
- Real PI names + institutions + USD amounts + program (PRCRP/BCRP/PCRP/etc.) + abstracts
- Distinct from the DoD USAspending ingest; provenance `dimensions_cdmrp` (dedicated source overrides aggregator rows via lower-priority-wins dedup)


## Step 1: Create Staging Table from S3

In [ ]:
%sql
-- Create the staging table from S3 parquet
CREATE OR REPLACE TABLE openalex.awards.cdmrp_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cdmrp/cdmrp_grants.parquet`;


In [ ]:
%sql
-- Check row count (should be ~23.7K)
SELECT COUNT(*) as total_projects FROM openalex.awards.cdmrp_raw;

In [ ]:
%sql
-- Inspect column names
DESCRIBE openalex.awards.cdmrp_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT * FROM openalex.awards.cdmrp_raw LIMIT 5;

## Step 2: Create CDMRP Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cdmrp_awards
USING delta
AS
WITH
cdmrp_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320338273  -- Congressionally Directed Medical Research Programs
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.short_abstract as description,
        f.funder_id,
        g.funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        COALESCE(NULLIF(TRIM(g.currency), ''), 'USD') as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        'grant' as funding_type,
        g.program as funder_scheme,
        'dimensions_cdmrp' as provenance,
        TRY_TO_DATE(SUBSTRING(g.start_date, 1, 10), 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(SUBSTRING(g.end_date, 1, 10), 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(SUBSTRING(g.start_date, 1, 10), 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(SUBSTRING(g.end_date, 1, 10), 'yyyy-MM-dd')) as end_year,
        CASE
            WHEN g.pi_last_name IS NOT NULL THEN
                struct(
                    INITCAP(g.pi_first_name) as given_name,
                    INITCAP(g.pi_last_name) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        COALESCE(g.institution, g.pi_affiliation) as name,
                        g.institution_country as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.cdmrp_raw g
    CROSS JOIN cdmrp_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'dimensions_cdmrp' AND priority = 237;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    237 as priority
FROM openalex.awards.cdmrp_awards;

## Verification Queries

In [ ]:
%sql
-- Check row count (should be ~23.7K)
SELECT COUNT(*) as total_awards FROM openalex.awards.cdmrp_awards;

In [ ]:
%sql
-- Sample the data
SELECT 
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    end_year
FROM openalex.awards.cdmrp_awards 
LIMIT 10;

In [ ]:
%sql
-- Check funder distribution (should all be CDMRP)
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.cdmrp_awards
GROUP BY funder.display_name
ORDER BY cnt DESC;

In [ ]:
%sql
-- Check program/scheme distribution
SELECT 
    funder_scheme,
    COUNT(*) as cnt,
    ROUND(SUM(amount)/1000000000, 2) as funding_billion_czk
FROM openalex.awards.cdmrp_awards
GROUP BY funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
-- Check year distribution
SELECT 
    start_year,
    COUNT(*) as cnt,
    ROUND(SUM(amount)/1000000, 1) as funding_millions_czk
FROM openalex.awards.cdmrp_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 20;

In [ ]:
%sql
-- Check data completeness
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_abstract,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_institution,
    COUNT(lead_investigator.affiliation.ids) as has_ror,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_with_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(SUM(amount)/1000000000, 2) as total_funding_billion_czk
FROM openalex.awards.cdmrp_awards;

In [ ]:
%sql
-- Check top institutions
SELECT 
    lead_investigator.affiliation.name as institution,
    COUNT(*) as cnt,
    ROUND(SUM(amount)/1000000000, 2) as funding_billion_czk
FROM openalex.awards.cdmrp_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY lead_investigator.affiliation.name
ORDER BY cnt DESC
LIMIT 20;

In [ ]:
%sql
-- Verify data was inserted into openalex_awards_raw
SELECT COUNT(*) as cdmrp_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'dimensions_cdmrp' AND priority = 237;